# Catchups - Taxonomy project

In [7]:
%load_ext kedro.ipython

The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


In [8]:
import pandas as pd
pd.set_option('max_colwidth', 300)

![image.png](image.png)

### **Updated Pipeline**

---

#### **Step 1: Input data preparation**
1. **Data inputs:**
   - Project abstracts or descriptions.
   - Extracted keywords (from DBPedia, RAKE, YAKE, KeyBERT, etc.).
   - Taxonomy labels, including hierarchical structure.

In [9]:
projects = catalog.load("gtr.projects.documents")

[02/18/25 11:26:11] INFO     Loading data from gtr.projects.documents (ParquetDataset)...       ]8;id=498223;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=152380;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\

In [10]:
projects.head(2)

,project_id,title,abstract_text,tech_abstract_text,potential_impact
0,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,Exploring the dark universe with quantum technologies,"The search to understand the nature of the elusive dark matter in our universe, making up 85% of its mass, is amongst the highest scientific priorities around the world. Terrestrial experiments focus efforts on Weakly Interacting Massive Particles (WIMPs) with ultra-low background experiments, o...",None,None
1,00022364-C7A7-4016-BEA5-29C8D160F674,Exploring the role of vitamin transport in insect models of disease vector biology,"Vector-borne diseases of plants, livestock and humans are infections transmitted by arthropods feeding on host plants or animals. These diseases have huge worldwide economic, social and health costs.\nMicronutrients such as vitamins are essential for all forms of animal life. However, many inver...",None,None



2. **Hierarchical concatenation of taxonomy labels:**
   - Transform the taxonomy into **hierarchically concatenated labels**:
     - For each bottom-level node $ l_j $, concatenate its parent labels to create a hierarchical path:
       $$
       l_j = \text{"Parent > Child > Bottom level"}
       $$

3. **Embeddings:**
   - Compute or retrieve embeddings for:
     - **Project abstracts** (document embeddings).
     - **Keywords** (keyword embeddings).
     - **Taxonomy labels** (concatenated hierarchical labels).

In [11]:
taxonomy = catalog.load("taxonomy.cwts.full.db")

[02/18/25 11:26:56] INFO     Loading data from taxonomy.cwts.full.db (ParquetDataset)...        ]8;id=319959;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=982751;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\

In [12]:
taxonomy.head(4)

,label,id_path,level,uuid
0,Physical Sciences,3,0,b35d6e90-36e2-537d-ac76-7d64b01b4c9d
1,Physical Sciences > Earth and Planetary Sciences,3 > 19,1,c43e0cd0-c84d-5094-93b3-337c899cbfcd
2,Physical Sciences > Earth and Planetary Sciences > Geophysics,3 > 19 > 1908,2,a3e52bd2-91fe-5b52-9811-d0ea3db295a0
3,Physical Sciences > Earth and Planetary Sciences > Geophysics > Tectonic and Geochronological Evolution of Orogens,3 > 19 > 1908 > 10001,3,2d79cb09-6eeb-54a1-b75a-832ad4e601a6



---

#### **Step 2: Keyword-taxonomy similarity**
1. **Keyword similarity:**
   - For each keyword $ k_i $, compute its similarity to all taxonomy labels $ l_j $:
     $$
     S_{\text{keywords}}(k_i, l_j) = \text{cosine\_similarity}(\text{embedding}(k_i), \text{embedding}(l_j))
     $$


2. **Shortlist labels for each keyword:**
   - Identify the top-$ k $ taxonomy labels $ \{l_{j_1}, l_{j_2}, \dots, l_{j_k}\} $ for each keyword, ranked by $ S_{\text{keywords}}(k_i, l_j) $.


---

#### **Step 3: Sentence-taxonomy similarity**
1. **Sentence similarity:**
   - Compute the similarity between the **project sentence embeddings** $ s_p $ for project $p$ and the shortlisted taxonomy labels $ \{l_{j_1}, \dots, l_{j_k}\} $:
     $$
     S_{\text{sentence}}(s_p, l_j) = \text{cosine\_similarity}(\text{embedding}(s_p), \text{embedding}(l_j))
     $$

2. **Normalisation for weights:**
   - For each project-label pair $(p, l_k)$, which appends all sentences $s_p$, we compute:

     - Mean similarity: $mean(p, l_k) = \frac{1}{n_p}\sum_{i=1}^{n_p} S_{\text{sentence}}(s_i, l_j)$

     - Maximum similarity: $max(p, l_k) = \max_i S_{\text{sentence}}(s_i, l_k)$

     - Match count: $count(p, l_k)$ = number of sentences matching label

   Combined score balancing frequency and strength:
   $$score(p, l_k) = \frac{1 + \log(1 + count(p, l_k))}{1 + \log(1 + n_p)} \cdot mean(p, l_k) \cdot (1 + \log(1 + max(p, l_k)))$$

  This prevents:
  - Short projects from being overly penalised
  - Long projects from dominating just due to length
  - Each project's strongest topic gets score 1.0
  - Other topics are scored relative to the strongest
  - Projects of different sizes can be compared fairly

   This approach recognises that:
  - A topic mentioned consistently across sentences is likely more relevant
  - But we shouldn't penalise focused projects that discuss fewer topics
  - Very strong individual matches should boost confidence
  - Final scores should reflect relative importance within each project

In [13]:
project_sentence_candidates = catalog.load("sentences.gtr_data.cwts_matches.intermediate")

                    INFO     Loading data from sentences.gtr_data.cwts_matches.intermediate     ]8;id=89828;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=445065;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\
                             (ParquetDataset)...                                                                   

In [14]:
project_sentence_candidates.head(20)

,project_id,sentence_id,taxonomy_label_id,taxonomy_label,similarity_score
0,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,619f10c0-3300-5261-a9b4-945ecccdd803,90fd84d0-d95b-5e78-89a2-19b31376c24d,Physical Sciences > Computer Science > Artificial Intelligence > Quantum Information and Computation,0.670365
1,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,619f10c0-3300-5261-a9b4-945ecccdd803,f1c393ec-eee6-5883-a7f1-52cdbc4b6874,Physical Sciences > Computer Science > Artificial Intelligence > Quantum Computing and Simulation,0.663663
2,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,619f10c0-3300-5261-a9b4-945ecccdd803,a78fe7e8-2a9e-58f2-b11d-8c956b88426e,Physical Sciences > Physics and Astronomy > Astronomy and Astrophysics > Search for Extraterrestrial Life and Intelligence,0.662514
3,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,619f10c0-3300-5261-a9b4-945ecccdd803,7df1e532-1e41-5df1-bf1d-1b4c1c06b59d,"Physical Sciences > Physics and Astronomy > Atomic and Molecular Physics, and Optics > Semiconductor Spintronics and Quantum Computing",0.659188
4,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,619f10c0-3300-5261-a9b4-945ecccdd803,3a333e07-18e4-5792-bb4f-4d100f7f35a4,"Physical Sciences > Physics and Astronomy > Atomic and Molecular Physics, and Optics > Quantum Dot Devices and Semiconductors",0.657013
5,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,93fc793f-5779-5ca6-9799-664daad47152,a78fe7e8-2a9e-58f2-b11d-8c956b88426e,Physical Sciences > Physics and Astronomy > Astronomy and Astrophysics > Search for Extraterrestrial Life and Intelligence,0.707222
6,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,93fc793f-5779-5ca6-9799-664daad47152,664144dd-8730-5844-be77-ba12b8ce592f,Physical Sciences > Physics and Astronomy > Nuclear and High Energy Physics > Particle Dark Matter and Detection Methods,0.688926
7,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,93fc793f-5779-5ca6-9799-664daad47152,132eb67b-d5b3-5b44-8707-095f8cd0398f,Physical Sciences > Physics and Astronomy > Astronomy and Astrophysics > Cosmological Parameters and Dark Energy,0.668843
8,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,93fc793f-5779-5ca6-9799-664daad47152,7ac6a930-ee4f-59eb-a4f5-81e309dd0865,"Physical Sciences > Physics and Astronomy > Atomic and Molecular Physics, and Optics > Magnetic Skyrmions and Spintronics",0.657319
9,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,93fc793f-5779-5ca6-9799-664daad47152,8146e49f-2ec1-579c-810a-b2db3c91d712,"Physical Sciences > Physics and Astronomy > Atomic and Molecular Physics, and Optics > Dusty Plasmas: Interdisciplinary Research Field",0.655224


   Filtering and normalisation:
   1. Project-level quantile filtering:
      $$S_{filtered}(p) = \{s : s > Q_q(S_{score}(p))\}$$
      where $Q_q(S_{score}(p))$ is the $q$-th quantile of scores within project $p$
      
   2. Per-project score normalisation:
      $$S_{final}(p, l_k) = \frac{S_{filtered}(p, l_k)}{\max_{l_j} S_{filtered}(p, l_j)}$$
      This ensures each project's top score is 1.0 while preserving relative strengths


---

#### **Step 4: Aggregate scores to labels**

  - For each keyword $k_j$, we have raw similarity scores with labels:
    $$sim(k_j, l_k) \text{ for } l_k \in L$$

  - Valid labels are restricted to those that passed sentence filtering:
    $$L_{valid}(p) = \{l_k : l_k \in S_{final}(p)\}$$

  - Keyword scores are then defined only on this subset:

    $$K_{score}(p, l_k) = \begin{cases}
      sim(k_j, l_k) & \text{if } l_k \in L_{valid}(p) \\
      \text{undefined} & \text{otherwise}
      \end{cases}$$

  This ensures keyword matches only reinforce labels that were relevant in sentence matching.

- Filter by keyword similarity quantile threshold:
  $$K_{filtered} = \{k : K_{score}(p, l_k) \geq Q_q(K_{score})\}$$
  
  This helps only consider keyword evidence that is sufficiently strong, avoiding noise from weak keyword matches.


In [16]:
project_scores = catalog.load("projects.gtr_data.cwts_scores.granular")

[02/18/25 11:29:16] INFO     Loading data from projects.gtr_data.cwts_scores.granular           ]8;id=291807;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=9278;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\
                             (ParquetDataset)...                                                                   

In [17]:
project_scores.head(5)

,project_id,sentence_id,taxonomy_label_id,taxonomy_label,similarity_score,similarity_score_global,similarity_score_key,sentence_score
0,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,93fc793f-5779-5ca6-9799-664daad47152,a78fe7e8-2a9e-58f2-b11d-8c956b88426e,Physical Sciences > Physics and Astronomy > Astronomy and Astrophysics > Search for Extraterrestrial Life and Intelligence,0.707222,0.675630,NaN,0.627022
1,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,93fc793f-5779-5ca6-9799-664daad47152,664144dd-8730-5844-be77-ba12b8ce592f,Physical Sciences > Physics and Astronomy > Nuclear and High Energy Physics > Particle Dark Matter and Detection Methods,0.688926,0.747938,0.682505,0.705987
2,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,8e9d3e94-443d-5327-9d97-c24ac14ae235,8146e49f-2ec1-579c-810a-b2db3c91d712,"Physical Sciences > Physics and Astronomy > Atomic and Molecular Physics, and Optics > Dusty Plasmas: Interdisciplinary Research Field",0.675209,0.693788,NaN,0.613262
3,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,2877838f-ad97-5faf-9ad9-3b6ae9672932,664144dd-8730-5844-be77-ba12b8ce592f,Physical Sciences > Physics and Astronomy > Nuclear and High Energy Physics > Particle Dark Matter and Detection Methods,0.701491,0.747938,0.682505,0.713527
4,00014AFD-3C1F-410E-8D00-8FF5A7F7AF54,2877838f-ad97-5faf-9ad9-3b6ae9672932,6cc5012e-da9e-5689-9b04-742a8b4dd3fb,Physical Sciences > Physics and Astronomy > Nuclear and High Energy Physics > Advancements in Particle Detector Technology,0.699583,0.704930,NaN,0.631229



---

#### **Step 5: Final label selection using relevance drop-off**

1. **Label relevance scores:**

   - - Initial weighted combination with configurable weights $\alpha$:
  $$score(p, l_k) = \big(\alpha \cdot S_{score}(p, l_k) + (1- \alpha) \cdot K_{filtered}(k_j, l_k)\big)^2$$



   - Aggregate to project-label level:

     - Take maximum of relevance scores

     - Count unique matching keywords

     - Average entropy across matches

2. **Global Binning**:

     - Use quantiles $Q_2$ and $Q_3$ across all scores:
       ```
       if score > Q_3:      high
       if Q_2 < score ≤ Q_3: medium
       if score ≤ Q_2:      low
       ```

3. **Local (Project-Level) Binning**:
     - For each project, compute relative gaps:
       $$gap_i = \frac{score(p, l_{(i)}) - score(p, l_{(i+1)})}{score(p, l_{(i)})}$$
     - Find two largest gaps ($i^*$, $j^*$)
     - Assign bins:
       ```
       if i ≤ i*:         high
       if i* < i ≤ j*:    medium
       if i > j*:         low
       ```

4. **Final Assignment**:
     - Take conservative approach:
       $$final\_bin = min(global\_bin, local\_bin)$$


## Checking an example

In [18]:
project_example_id = "00014AFD-3C1F-410E-8D00-8FF5A7F7AF54"
project_example_id2 = "51709B42-E59C-436E-8D6B-906BFC9086E5"

In [19]:
print(projects[projects["project_id"] == project_example_id]["title"].iloc[0])
print("Abstract:")
print(projects[projects["project_id"] == project_example_id]["abstract_text"].iloc[0])

Exploring the dark universe with quantum technologies
Abstract:
The search to understand the nature of the elusive dark matter in our universe, making up 85% of its mass, is amongst the highest scientific priorities around the world. Terrestrial experiments focus efforts on Weakly Interacting Massive Particles (WIMPs) with ultra-low background experiments, operating many tonnes of target mass in deep underground sites. Such experiments have swept the bulk of the available electroweak parameter space for WIMPs and in the next decade will approach an irreducible background from coherent scattering of neutrinos - indistinguishable from WIMPs. Internationally, efforts are ramping up to explore new avenues towards the first definitive detection of galactic dark matter: quantum technologies is an exciting new frontier that may provide the breakthrough.

Levitating nanospheres, opto-mechanically held with high precision, represent targets with unprecedented sensitivity to dark matter scatteri

In [20]:
print(projects[projects["project_id"] == project_example_id2]["title"].iloc[0])
print("Abstract:")
print(projects[projects["project_id"] == project_example_id2]["abstract_text"].iloc[0])

Innovative Packaging
Abstract:
The project aim is to create an innovative packaging solution that is both environmentally friendly and boosts the commercial appeal of our new range of flavoured waters.


##### **CWTS Topics**

In [21]:
project_scores_cwts = catalog.load("projects.gtr_data.cwts_scores.zeroshot")

[02/18/25 11:29:33] INFO     Loading data from projects.gtr_data.cwts_scores.zeroshot           ]8;id=962814;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=312736;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\
                             (ParquetDataset)...                                                                   

In [37]:
project_example_cwts = project_scores_cwts[
    project_scores_cwts["project_id"] == project_example_id
]
project_example_cwts.sort_values(by="zeroshot_score", ascending=False).head(10)[
    [
        "taxonomy_label",
        "relevance_score",
        "global_bin",
        "local_bin",
        "confidence_bin",
        "zeroshot_score",
        "zeroshot_bin",
    ]
]

,taxonomy_label,relevance_score,global_bin,local_bin,confidence_bin,zeroshot_score,zeroshot_bin
1,Physical Sciences > Physics and Astronomy > Nuclear and High Energy Physics > Particle Dark Matter and Detection Methods,0.428339,high,high,high,0.840717,high
5,Physical Sciences > Computer Science > Artificial Intelligence > Quantum Information and Computation,0.135478,high,medium,medium,0.652985,medium
3,"Physical Sciences > Physics and Astronomy > Atomic and Molecular Physics, and Optics > Semiconductor Spintronics and Quantum Computing",0.137634,high,medium,medium,0.638502,medium
6,"Physical Sciences > Physics and Astronomy > Atomic and Molecular Physics, and Optics > Cavity Optomechanics and Nanomechanical Systems",0.122834,medium,low,low,0.567757,medium
2,Physical Sciences > Physics and Astronomy > Nuclear and High Energy Physics > Advancements in Particle Detector Technology,0.126246,medium,low,low,0.245854,very low
4,"Physical Sciences > Physics and Astronomy > Atomic and Molecular Physics, and Optics > Dusty Plasmas: Interdisciplinary Research Field",0.122652,medium,low,low,0.121581,very low
7,Physical Sciences > Physics and Astronomy > Astronomy and Astrophysics > Search for Extraterrestrial Life and Intelligence,0.125404,medium,low,low,0.057918,very low
0,"Physical Sciences > Physics and Astronomy > Atomic and Molecular Physics, and Optics > Quantum Dot Devices and Semiconductors",0.139777,high,medium,medium,0.053206,very low


In [38]:
project_example2_cwts = project_scores_cwts[project_scores_cwts["project_id"] == project_example_id2]
project_example2_cwts[    [
        "taxonomy_label",
        "relevance_score",
        "global_bin",
        "local_bin",
        "confidence_bin",
        "zeroshot_score",
        "zeroshot_bin",
    ]].sort_values(by="zeroshot_score", ascending=False).head(10)

,taxonomy_label,relevance_score,global_bin,local_bin,confidence_bin,zeroshot_score,zeroshot_bin
473593,"Social Sciences > Business, Management and Accounting > Marketing > Role of Packaging Design in Consumer Behavior",0.68047,high,high,high,0.368633,low


##### **OpenAlex Concepts**

In [39]:
# taxonomy_concepts = catalog.load("taxonomy.oa_concepts.full.db")
project_scores_concepts = catalog.load("projects.gtr_data.oa_concepts_scores.zeroshot")

[02/18/25 11:36:35] INFO     Loading data from projects.gtr_data.oa_concepts_scores.zeroshot    ]8;id=179238;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=722040;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\
                             (ParquetDataset)...                                                                   

In [40]:
# consider the same example
project_example_concepts = project_scores_concepts[
    project_scores_concepts["project_id"] == project_example_id
]
project_example_concepts[
    [
        "taxonomy_label",
        "relevance_score",
        "global_bin",
        "local_bin",
        "confidence_bin",
        "zeroshot_score",
        "zeroshot_bin",
    ]
].sort_values(by="zeroshot_score", ascending=False).head(10)

,taxonomy_label,relevance_score,global_bin,local_bin,confidence_bin,zeroshot_score,zeroshot_bin
5,Physics > Astrophysics > Dark matter,0.151598,high,high,high,0.991067,very high
3,Physics > Astronomy > Dark matter,0.154205,high,high,high,0.990292,very high
6,Physics > Particle physics > Dark matter,0.152257,high,high,high,0.986979,very high
4,Physics > Astronomy > Dark matter > WIMP,0.134293,high,high,high,0.954943,very high
2,Physics > Particle physics > Dark matter > WIMP,0.131511,high,medium,medium,0.920861,very high
1,Physics > Optics > Detector > Scintillation > Alpha-particle spectroscopy,0.088784,low,low,low,0.898944,high
13,Physics > Quantum mechanics > Cosmology > Dark energy > Dark radiation,0.130718,high,medium,medium,0.861072,high
8,Physics > Optics > Detector > Scintillation counter > Alpha-particle spectroscopy,0.087797,low,low,low,0.849909,high
9,Physics > Quantum mechanics > Cosmology > Dark energy,0.129840,high,low,low,0.843251,high
11,Physics > Quantum mechanics > Optical computing,0.100973,low,low,low,0.755840,high


In [41]:
# consider the same packaging example
project_example2_concepts = project_scores_concepts[
    project_scores_concepts["project_id"] == project_example_id2
]
project_example2_concepts[
    [
        "taxonomy_label",
        "relevance_score",
        "global_bin",
        "local_bin",
        "confidence_bin",
        "zeroshot_score",
        "zeroshot_bin",
    ]
].sort_values(by="zeroshot_score", ascending=False).head(10)

,taxonomy_label,relevance_score,global_bin,local_bin,confidence_bin,zeroshot_score,zeroshot_bin


##### **GOScience Concepts**

In [28]:
# taxonomy_goscience = catalog.load("taxonomy.goscience.full.db")
project_scores_goscience = catalog.load("projects.gtr_data.goscience_scores.zeroshot")

[02/18/25 11:31:45] INFO     Loading data from projects.gtr_data.goscience_scores.zeroshot      ]8;id=815762;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=751206;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\
                             (ParquetDataset)...                                                                   

In [42]:
# consider the same example
project_example_goscience = project_scores_goscience[
    project_scores_goscience["project_id"] == project_example_id
]
project_example_goscience[
    [
        "taxonomy_label",
        "relevance_score",
        "global_bin",
        "local_bin",
        "confidence_bin",
        "zeroshot_score",
        "zeroshot_bin",
    ]
].sort_values(by="zeroshot_score", ascending=False).head(10)

,taxonomy_label,relevance_score,global_bin,local_bin,confidence_bin,zeroshot_score,zeroshot_bin
8,Quantum technologies,0.264941,high,medium,medium,0.987496,very high
12,Quantum technologies > Quantum sensors,0.629353,high,high,high,0.826081,high
0,Electronics and photonics > Sensors > Quantum sensors,0.216247,high,medium,medium,0.669259,medium
10,Advanced materials and manufacturing > Nanotechnology,0.107488,high,low,low,0.622230,medium
1,Future computing > Future computing paradigms > Quantum computing,0.124121,high,low,low,0.592637,medium
15,Advanced materials and manufacturing > Nanotechnology > Nanomachines,0.109546,high,low,low,0.539083,medium
11,Quantum technologies > Quantum imaging,0.374392,high,medium,medium,0.505615,medium
5,Quantum technologies > Quantum computing,0.263342,high,medium,medium,0.355439,low
3,Advanced materials and manufacturing > Nanotechnology > High NA EUV,0.106993,high,low,low,0.222362,very low
6,Quantum technologies > Quantum communications,0.125818,high,low,low,0.197367,very low


In [43]:
# consider the same second example
project_example2_goscience = project_scores_goscience[
    project_scores_goscience["project_id"] == project_example_id2
]
project_example2_goscience[
    [
        "taxonomy_label",
        "relevance_score",
        "global_bin",
        "local_bin",
        "confidence_bin",
        "zeroshot_score",
        "zeroshot_bin",
    ]
].sort_values(by="zeroshot_score", ascending=False).head(10)

,taxonomy_label,relevance_score,global_bin,local_bin,confidence_bin,zeroshot_score,zeroshot_bin
409449,Electronics and photonics > Novel electronics,0.452569,low,low,low,0.027247,very low
409450,Energy > Novel batteries > Alternative batteries,0.607808,medium,low,low,0.005609,very low
409448,Energy > Novel batteries,0.638141,high,high,high,0.003057,very low


---

#### **Step 6: OpenAI API Validation**  

1. **Algorithm-Agnostic Expert Labels**  
   - The LLM evaluates projects **independently** of algorithm results.  
   - Uses **RAG** to access full taxonomy context.  
   - Assigns confidence levels (**high/medium/low**) to chosen labels.  
   - Provides a **baseline set of “true” labels** unbiased by algorithm choices.  

2. **Algorithm Result Evaluation**  
   - The LLM evaluates **labels proposed by the matching algorithm**.  
   - Performs **binary classification** (`true` or `false positive`) with explanations.  
   - Identifies **algorithmic errors and biases** by detecting misleading matches.  
   - Provides **direct feedback** on algorithm performance.  

##### **Performance Measurement**  
By combining both evaluations, we can identify:  
- **True Positives** → The algorithm proposes a **high-confidence label** that the expert agrees with.  
- **False Positives** → The algorithm proposes a **high-confidence label** that the expert rejects.  
- **False Negatives** → The expert identifies a **high-confidence label** that the algorithm missed.  

This helps estimate **precision, recall, and F1 scores** for parameter tuning.  

##### **Parameter Space**  
Key algorithmic parameters being tuned:  
- **Weighting balance** → Relative importance of **sentence-level vs. keyword-level** matching.  
- **Similarity thresholds** → Minimum required **similarity scores**.  
- **Confidence thresholds** →  
  - **Global:** Dataset-wide similarity cutoffs.  
  - **Local:** Project-specific thresholds, accounting for relative label strengths.  

In [31]:
validation_example_id = "DA007068-2FB3-477E-95BE-D0C276B1AF87"

In [32]:
print(projects[projects["project_id"] == validation_example_id]["title"].iloc[0])
print("Abstract:")
print(projects[projects["project_id"] == validation_example_id]["abstract_text"].iloc[0])

Between protection and exclusion: Separated child migrants' care relationships and caring practices
Abstract:
The promise of this project lies in generating knowledge that both analyses and provides ways to address one of the greatest global challenges of our time: the care and well-being of children affected by transnational displacement and migration. It will offer insights into the care of separated migrant children in England, starting from the premise that care is not necessarily limited to that provided by an adult or the state. Our pilot studies demonstrate that a crucial way separated migrant children survive the challenges of migration and settlement is through the care they provide and receive from other migrant children. Using creative research methods designed to involve separated migrant children and adult stakeholders in reflecting on their understandings and experiences of care, this project will not only point to 'cracks in the system' (Rosen et al., 2017) but offer ins

In [33]:
agnostic_labels = catalog.load("tuning.cwts.expert_labels.processed")

[02/18/25 11:31:52] INFO     Loading data from tuning.cwts.expert_labels.processed              ]8;id=439102;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=698968;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\
                             (ParquetDataset)...                                                                   

In [ ]:
agnostic_labels_example = agnostic_labels[agnostic_labels["project_id"] == validation_example_id]

# sort likelihood by "high", "medium", "low" (note needs numeric mapping)
# Create numeric mapping for sorting
likelihood_map = {"high": 3, "medium": 2, "low": 1}

# Sort using the mapping
agnostic_labels_example = agnostic_labels_example.assign(
    likelihood_num=agnostic_labels_example["likelihood"].map(likelihood_map)
).sort_values("likelihood_num", ascending=False).drop("likelihood_num", axis=1)

display(agnostic_labels_example)

,project_id,taxonomy_label,likelihood,taxonomy_label_id
1403,DA007068-2FB3-477E-95BE-D0C276B1AF87,Social Sciences > Social Sciences > Law > Child Protection and Legal Frameworks,high,1ce35fbc-f0c1-5952-a559-0e610d396200
1404,DA007068-2FB3-477E-95BE-D0C276B1AF87,"Social Sciences > Social Sciences > Demography > Migration, Education, and Policy Implications",high,636e283f-092a-5888-981b-55a1868ccaf5
1410,DA007068-2FB3-477E-95BE-D0C276B1AF87,Social Sciences > Social Sciences > Sociology and Political Science > Participation of Children in Research and Policy,high,3e3ce970-7144-5589-9195-643f01c7b034
1401,DA007068-2FB3-477E-95BE-D0C276B1AF87,Social Sciences > Social Sciences > Safety Research > Child Welfare and Foster Care System,medium,65c70378-3a52-52c0-9cb2-8b3a66b9e70c
1405,DA007068-2FB3-477E-95BE-D0C276B1AF87,"Health Sciences > Medicine > Public Health, Environmental and Occupational Health > Ethical Considerations in Medical Research Participation",medium,cd5c4a8c-26d0-5843-8509-33b36268d5a7
1406,DA007068-2FB3-477E-95BE-D0C276B1AF87,Social Sciences > Social Sciences > Sociology and Political Science > Secondary Analysis of Qualitative Data,medium,fba37c34-03f5-58b3-9192-dcee9e94e36d
1407,DA007068-2FB3-477E-95BE-D0C276B1AF87,Social Sciences > Social Sciences > Health > Impact of International Migration on Public Health,medium,34a5f90a-e262-59b5-8f14-a3312928915f
1408,DA007068-2FB3-477E-95BE-D0C276B1AF87,"Health Sciences > Medicine > Pediatrics, Perinatology and Child Health > Global Maternal and Child Health Outcomes",low,ca2b5f53-862f-545c-bc22-9f088cc61689
1409,DA007068-2FB3-477E-95BE-D0C276B1AF87,"Social Sciences > Social Sciences > Geography, Planning and Development > Qualitative Research in Tourism",low,166ca610-c4a8-5447-8eaf-2af2b9c7ee59


In [45]:
algorithm_labels = catalog.load("tuning.cwts.expert_assessment.processed")

[02/18/25 11:38:37] INFO     Loading data from tuning.cwts.expert_assessment.processed          ]8;id=582903;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=601256;file:///home/dampudia/miniconda3/envs/.dsit/lib/python3.12/site-packages/kedro/io/data_catalog.py#389\389]8;;\
                             (ParquetDataset)...                                                                   

In [65]:
algorithm_labels_example = algorithm_labels[algorithm_labels["project_id"] == validation_example_id].sort_values(by="positive", ascending=False)
algorithm_labels_example.head(10)

,project_id,taxonomy_label_id,positive,explanation,taxonomy_label
2216,DA007068-2FB3-477E-95BE-D0C276B1AF87,1ce35fbc-f0c1-5952-a559-0e610d396200,True,"The project deals with protected migrant children and is related to child protection and legal frameworks, thus the label is compatible.",Social Sciences > Social Sciences > Law > Child Protection and Legal Frameworks
2219,DA007068-2FB3-477E-95BE-D0C276B1AF87,65c70378-3a52-52c0-9cb2-8b3a66b9e70c,True,"The research involves care and welfare challenges of separated migrant children, hence the taxonomy about child welfare and foster care system is relevant.",Social Sciences > Social Sciences > Safety Research > Child Welfare and Foster Care System
2220,DA007068-2FB3-477E-95BE-D0C276B1AF87,7cf90a13-5e42-5997-8421-b23825a5a95b,True,"The project involves separated migrant children which fits under the theme of migration studies, hence the taxonomy label is applicable.",Social Sciences > Social Sciences > Demography > Transnational Diasporas and Migration Studies
2223,DA007068-2FB3-477E-95BE-D0C276B1AF87,b51fbe3c-bf96-5195-a545-b95008d5a520,True,"The project is based on observations and experiences of separated child migrants and adult stakeholders, implying a form of participatory action research, therefore this label is appropriate.",Social Sciences > Social Sciences > Education > Participatory Action Research in Education and Social Sciences
2215,DA007068-2FB3-477E-95BE-D0C276B1AF87,048ca1a5-2029-532c-a70f-09370195accf,False,"The project focuses on the care of separated migrant children in the England rather than the French Territories, hence the label is not applicable.","Social Sciences > Social Sciences > Sociology and Political Science > Migration, Health, and Inequality in French Territories"
2217,DA007068-2FB3-477E-95BE-D0C276B1AF87,34a5f90a-e262-59b5-8f14-a3312928915f,False,"The project concentrates on the wellbeing of separated migrant children without a specific focus on public health impacts, hence the label is not suitable.",Social Sciences > Social Sciences > Health > Impact of International Migration on Public Health
2218,DA007068-2FB3-477E-95BE-D0C276B1AF87,636e283f-092a-5888-981b-55a1868ccaf5,False,"The project discusses the care of separated migrant children but does not dive deep into education and policy implications connected with migration. Thus, the label is inappropriate.","Social Sciences > Social Sciences > Demography > Migration, Education, and Policy Implications"
2221,DA007068-2FB3-477E-95BE-D0C276B1AF87,883af7e9-4ca6-5074-bb14-59fe25d2f6e8,False,"The project doesn't explicitly address health and healthcare disparities. Instead, the focus is on migrant children's care and wellbeing.",Social Sciences > Social Sciences > Health > Social Determinants of Health and Healthcare Disparities
2222,DA007068-2FB3-477E-95BE-D0C276B1AF87,92cfe6c3-a01a-51d7-8bbb-49311611b03f,False,"The project does not entail specific focus on the mental health of refugees and immigrants, thus the taxonomy label is not suitable.",Social Sciences > Psychology > Clinical Psychology > Mental Health of Refugees and Immigrants
2224,DA007068-2FB3-477E-95BE-D0C276B1AF87,f37f58dd-899c-50b5-bfb1-cc2dbbd5563a,False,"The project does not analyse child injury preventions globally but rather discusses separated migrant children's care in England, so this label is incorrect.","Health Sciences > Medicine > Public Health, Environmental and Occupational Health > Global Burden of Child Injury Prevention"


In [48]:
validation_example_cwts = project_scores_cwts[
    project_scores_cwts["project_id"] == validation_example_id
]
validation_example_cwts[
    [
        "taxonomy_label",
        "relevance_score",
        "global_bin",
        "local_bin",
        "confidence_bin",
        "zeroshot_score",
        "zeroshot_bin",
    ]
].sort_values(by="zeroshot_score", ascending=False).head(20)

,taxonomy_label,relevance_score,global_bin,local_bin,confidence_bin,zeroshot_score,zeroshot_bin
1258987,"Social Sciences > Social Sciences > Demography > Migration, Education, and Policy Implications",0.078122,low,medium,low,0.852752,high
1258989,Social Sciences > Social Sciences > Demography > Transnational Diasporas and Migration Studies,0.159529,low,high,low,0.626381,medium
1258988,Social Sciences > Social Sciences > Safety Research > Child Welfare and Foster Care System,0.079978,low,medium,low,0.381832,low
1258985,Social Sciences > Social Sciences > Law > Child Protection and Legal Frameworks,0.068034,low,low,low,0.317889,low
1258992,Social Sciences > Social Sciences > Education > Participatory Action Research in Education and Social Sciences,0.067657,low,low,low,0.161188,very low
1258990,Social Sciences > Social Sciences > Health > Social Determinants of Health and Healthcare Disparities,0.069776,low,low,low,0.043855,very low
1258986,Social Sciences > Social Sciences > Health > Impact of International Migration on Public Health,0.078628,low,medium,low,0.027928,very low
1258991,Social Sciences > Psychology > Clinical Psychology > Mental Health of Refugees and Immigrants,0.068182,low,low,low,0.022095,very low
1258993,"Health Sciences > Medicine > Public Health, Environmental and Occupational Health > Global Burden of Child Injury Prevention",0.080469,low,medium,low,0.002521,very low
1258984,"Social Sciences > Social Sciences > Sociology and Political Science > Migration, Health, and Inequality in French Territories",0.067530,low,low,low,0.001658,very low


In [68]:
# Merge the three sets of results
merged_results = (
    # Start with agnostic labels
    agnostic_labels_example[["taxonomy_label", "likelihood"]]
    # Merge with algorithm labels
    .merge(
        algorithm_labels_example[["taxonomy_label", "positive"]], 
        on="taxonomy_label",
        how="outer"
    )
    # Merge with zeroshot scores
    .merge(
        validation_example_cwts[["taxonomy_label", "confidence_bin", "zeroshot_bin"]],
        on="taxonomy_label", 
        how="outer"
    )
)

likelihood_map = {"high": 3, "medium": 2, "low": 1}
merged_results = merged_results.assign(
    likelihood_num=merged_results["likelihood"].map(likelihood_map)
).sort_values(["likelihood_num", "positive"], ascending=[False, False]).drop("likelihood_num", axis=1)

display(merged_results)

,taxonomy_label,likelihood,positive,confidence_bin,zeroshot_bin
10,Social Sciences > Social Sciences > Law > Child Protection and Legal Frameworks,high,True,low,low
4,"Social Sciences > Social Sciences > Demography > Migration, Education, and Policy Implications",high,False,low,high
13,Social Sciences > Social Sciences > Sociology and Political Science > Participation of Children in Research and Policy,high,NaN,NaN,NaN
11,Social Sciences > Social Sciences > Safety Research > Child Welfare and Foster Care System,medium,True,low,low
8,Social Sciences > Social Sciences > Health > Impact of International Migration on Public Health,medium,False,low,very low
1,"Health Sciences > Medicine > Public Health, Environmental and Occupational Health > Ethical Considerations in Medical Research Participation",medium,NaN,NaN,NaN
14,Social Sciences > Social Sciences > Sociology and Political Science > Secondary Analysis of Qualitative Data,medium,NaN,NaN,NaN
0,"Health Sciences > Medicine > Pediatrics, Perinatology and Child Health > Global Maternal and Child Health Outcomes",low,NaN,NaN,NaN
7,"Social Sciences > Social Sciences > Geography, Planning and Development > Qualitative Research in Tourism",low,NaN,NaN,NaN
5,Social Sciences > Social Sciences > Demography > Transnational Diasporas and Migration Studies,NaN,True,low,medium


#### Another example

In [56]:
validation_example2 = "22028E13-05C7-439D-91D1-C88D2249A8DF"

In [57]:
print(projects[projects["project_id"] == validation_example2]["title"].iloc[0])
print("Abstract:")
print(projects[projects["project_id"] == validation_example2]["abstract_text"].iloc[0])

Fisheries governance in an inclusive and sustainable blue economy
Abstract:
The blue economy is expected to double in value to USD 3 trillion by 2030, globally. Rapid expansion of diverse sectors (aquaculture, coastal tourism, shipping, mining and offshore renewables) can displace or dispossess 'traditional' maritime sectors of the ocean resources they depend upon. In particular, (small-scale) fisheries are &quot;subtly and overtly squeezed for geographic, political and economic space&quot; with important implications for governance processes (e.g., trust, participation and compliance) and outcomes (livelihoods, food security and sustainable management of resources). In the UK, there is strong growth predicted in offshore renewables and aquaculture sectors alongside expanding conservation and protection of marine space through, for example, Marine Conservation Zones and Highly Protected Marine Areas all of which is escalating tensions within fisheries and marine governance. This doctor

In [69]:
agnostic_labels_example2 = agnostic_labels[agnostic_labels["project_id"] == validation_example2]

# sort likelihood by "high", "medium", "low" (note needs numeric mapping)
# Create numeric mapping for sorting
likelihood_map = {"high": 3, "medium": 2, "low": 1}

# Sort using the mapping
agnostic_labels_example2 = agnostic_labels_example2.assign(
    likelihood_num=agnostic_labels_example2["likelihood"].map(likelihood_map)
).sort_values("likelihood_num", ascending=False).drop("likelihood_num", axis=1)

display(agnostic_labels_example2)

,project_id,taxonomy_label,likelihood,taxonomy_label_id
206,22028E13-05C7-439D-91D1-C88D2249A8DF,"Physical Sciences > Environmental Science > Management, Monitoring, Policy and Law > Importance of Marine Spatial Planning in Ecosystem Management",high,fc52a1b0-54c6-55ea-a462-c91081420ecf
207,22028E13-05C7-439D-91D1-C88D2249A8DF,Physical Sciences > Environmental Science > Global and Planetary Change > Impact of Aquaculture on Marine Ecosystems and Food Supply,high,458926e0-7cac-561a-bef6-9c2b9b8c14ab
208,22028E13-05C7-439D-91D1-C88D2249A8DF,Physical Sciences > Environmental Science > Ecology > Human Impact on Marine Ecology and Fisheries,medium,8deffcad-1d23-5bef-88d7-62cec1f7cef8
209,22028E13-05C7-439D-91D1-C88D2249A8DF,"Social Sciences > Social Sciences > Geography, Planning and Development > Geography and Public Policy in Academic Discourse",medium,c45570b1-2cf4-5c22-8832-8040cda717d6
205,22028E13-05C7-439D-91D1-C88D2249A8DF,Social Sciences > Social Sciences > Sociology and Political Science > Arctic Shipping and Governance,low,5aa7bf81-ba22-5774-9f8e-7f7b79cdab30
210,22028E13-05C7-439D-91D1-C88D2249A8DF,"Physical Sciences > Environmental Science > Management, Monitoring, Policy and Law > Marine Genetic Resources and International Law",low,a55d4a1a-de6a-5707-aa30-e380c31ea392


In [70]:
algorithm_labels_example2 = algorithm_labels[
    algorithm_labels["project_id"] == validation_example2
].sort_values(by="positive", ascending=False)
display(algorithm_labels_example2)

,project_id,taxonomy_label_id,positive,explanation,taxonomy_label
465,22028E13-05C7-439D-91D1-C88D2249A8DF,458926e0-7cac-561a-bef6-9c2b9b8c14ab,True,"The research mentions the impact of aquaculture sectors on fisheries and marine governance, which can affect marine ecosystems and food supply.",Physical Sciences > Environmental Science > Global and Planetary Change > Impact of Aquaculture on Marine Ecosystems and Food Supply
466,22028E13-05C7-439D-91D1-C88D2249A8DF,8deffcad-1d23-5bef-88d7-62cec1f7cef8,True,This research directly addresses the human impact on marine ecology and fisheries due to the expansion of various sectors within the blue economy.,Physical Sciences > Environmental Science > Ecology > Human Impact on Marine Ecology and Fisheries
467,22028E13-05C7-439D-91D1-C88D2249A8DF,9264e43f-e038-59d9-a20f-fdbc6d16dc66,True,The project aims to understand experiences and responses to spatial squeeze in the UK's blue economy and seeks to identify methods of conflict management and negotiation strategies in fisheries and marine governance.,Social Sciences > Social Sciences > Sociology and Political Science > Conflict Management and Negotiation Strategies
472,22028E13-05C7-439D-91D1-C88D2249A8DF,c621b9f8-18e1-5b8f-8488-00b2cf8e29d7,True,This project is built around enhancing local peacebuilding and resilience governance. A focus of the research is to improve collaboration and trust in fisheries and marine governance in order to address root causes of existing conflicts.,Social Sciences > Social Sciences > Sociology and Political Science > Local Turn in Peacebuilding and Resilience Governance
473,22028E13-05C7-439D-91D1-C88D2249A8DF,fc52a1b0-54c6-55ea-a462-c91081420ecf,True,The phenomenon of 'spatial squeeze' and displacement mentioned in the project directly pertains to the necessity of marine spatial planning in eco-system management.,"Physical Sciences > Environmental Science > Management, Monitoring, Policy and Law > Importance of Marine Spatial Planning in Ecosystem Management"
464,22028E13-05C7-439D-91D1-C88D2249A8DF,39b36457-2a96-565e-9931-4a2a949ffb75,False,"While the project is broadly about the impact on fisheries, it does not mention climate change directly as a factor in the research.",Physical Sciences > Environmental Science > Global and Planetary Change > Impacts of Climate Change on Marine Fisheries
468,22028E13-05C7-439D-91D1-C88D2249A8DF,a55d4a1a-de6a-5707-aa30-e380c31ea392,False,The research does not mention studying marine genetic resources or international law.,"Physical Sciences > Environmental Science > Management, Monitoring, Policy and Law > Marine Genetic Resources and International Law"
469,22028E13-05C7-439D-91D1-C88D2249A8DF,a7880906-d893-5101-9d7a-afcdc5ee62f7,False,There is no mention about the Cyprus conflict or peacebuilding related to it. The conflict in this research is concerning the use of marine resources.,Social Sciences > Social Sciences > Sociology and Political Science > Root Causes and Peacebuilding in Cyprus Conflict
470,22028E13-05C7-439D-91D1-C88D2249A8DF,aecf4154-190e-5dda-9ad1-c7ab3db64465,False,"While the project discusses marine policy and governance, it does not specifically address ship recycling or offshore decommissioning.",Physical Sciences > Engineering > Ocean Engineering > Ship Recycling and Offshore Decommissioning
471,22028E13-05C7-439D-91D1-C88D2249A8DF,ba8eaa47-b4ac-5e37-88f7-649e93ab5dde,False,"Innovative mining technology nor sustainable development related to it is mentioned in the project, thus this label does not fit.",Physical Sciences > Engineering > Ocean Engineering > Innovative Mining Technology and Sustainable Development


In [71]:
validation_example_cwts2 = project_scores_cwts[
    project_scores_cwts["project_id"] == validation_example2
]
validation_example_cwts2[
    [
        "taxonomy_label",
        "relevance_score",
        "global_bin",
        "local_bin",
        "confidence_bin",
        "zeroshot_score",
        "zeroshot_bin",
    ]
].sort_values(by="zeroshot_score", ascending=False).head(20)

,taxonomy_label,relevance_score,global_bin,local_bin,confidence_bin,zeroshot_score,zeroshot_bin
197060,Physical Sciences > Environmental Science > Ecology > Human Impact on Marine Ecology and Fisheries,0.611949,high,high,high,0.831280,high
197061,Social Sciences > Social Sciences > Sociology and Political Science > Conflict Management and Negotiation Strategies,0.053249,low,low,low,0.541409,medium
197062,"Physical Sciences > Environmental Science > Management, Monitoring, Policy and Law > Marine Genetic Resources and International Law",0.084117,medium,low,low,0.327883,low
197059,Physical Sciences > Environmental Science > Global and Planetary Change > Impact of Aquaculture on Marine Ecosystems and Food Supply,0.172430,high,medium,medium,0.249813,very low
197065,Physical Sciences > Engineering > Ocean Engineering > Innovative Mining Technology and Sustainable Development,0.060171,low,low,low,0.164920,very low
197066,Social Sciences > Social Sciences > Sociology and Political Science > Local Turn in Peacebuilding and Resilience Governance,0.051401,low,low,low,0.104569,very low
197067,"Physical Sciences > Environmental Science > Management, Monitoring, Policy and Law > Importance of Marine Spatial Planning in Ecosystem Management",0.521885,high,high,high,0.096706,very low
197058,Physical Sciences > Environmental Science > Global and Planetary Change > Impacts of Climate Change on Marine Fisheries,0.156302,medium,medium,medium,0.067362,very low
197064,Physical Sciences > Engineering > Ocean Engineering > Ship Recycling and Offshore Decommissioning,0.059436,low,low,low,0.006890,very low
197063,Social Sciences > Social Sciences > Sociology and Political Science > Root Causes and Peacebuilding in Cyprus Conflict,0.051011,low,low,low,0.002059,very low


In [72]:
# Merge the three sets of results
merged_results = (
    # Start with agnostic labels
    agnostic_labels_example2[["taxonomy_label", "likelihood"]]
    # Merge with algorithm labels
    .merge(
        algorithm_labels_example2[["taxonomy_label", "positive"]], 
        on="taxonomy_label",
        how="outer"
    )
    # Merge with zeroshot scores
    .merge(
        validation_example_cwts2[["taxonomy_label", "confidence_bin", "zeroshot_bin"]],
        on="taxonomy_label", 
        how="outer"
    )
)

likelihood_map = {"high": 3, "medium": 2, "low": 1}
merged_results = merged_results.assign(
    likelihood_num=merged_results["likelihood"].map(likelihood_map)
).sort_values(["likelihood_num", "positive"], ascending=[False, False]).drop("likelihood_num", axis=1)

display(merged_results)

,taxonomy_label,likelihood,positive,confidence_bin,zeroshot_bin
3,Physical Sciences > Environmental Science > Global and Planetary Change > Impact of Aquaculture on Marine Ecosystems and Food Supply,high,True,medium,very low
5,"Physical Sciences > Environmental Science > Management, Monitoring, Policy and Law > Importance of Marine Spatial Planning in Ecosystem Management",high,True,high,very low
2,Physical Sciences > Environmental Science > Ecology > Human Impact on Marine Ecology and Fisheries,medium,True,high,high
7,"Social Sciences > Social Sciences > Geography, Planning and Development > Geography and Public Policy in Academic Discourse",medium,NaN,NaN,NaN
6,"Physical Sciences > Environmental Science > Management, Monitoring, Policy and Law > Marine Genetic Resources and International Law",low,False,low,low
8,Social Sciences > Social Sciences > Sociology and Political Science > Arctic Shipping and Governance,low,NaN,NaN,NaN
9,Social Sciences > Social Sciences > Sociology and Political Science > Conflict Management and Negotiation Strategies,NaN,True,low,medium
10,Social Sciences > Social Sciences > Sociology and Political Science > Local Turn in Peacebuilding and Resilience Governance,NaN,True,low,very low
0,Physical Sciences > Engineering > Ocean Engineering > Innovative Mining Technology and Sustainable Development,NaN,False,low,very low
1,Physical Sciences > Engineering > Ocean Engineering > Ship Recycling and Offshore Decommissioning,NaN,False,low,very low


#### Another example

In [76]:
validation_example3 = "113BF950-3519-486A-A642-8578B742B755"

In [77]:
print(projects[projects["project_id"] == validation_example3]["title"].iloc[0])
print("Abstract:")
print(projects[projects["project_id"] == validation_example3]["abstract_text"].iloc[0])

Measuring life quality from digital footprints for informed policy decision making
Abstract:
This project aims to develop new measures of life quality from digital footprints (i.e., Internet and social media data). It fits nicely with the DCT pathway. It will employ interdisciplinary approaches from psychology, management, and data science to examine the changing nature of data usage and how emerging digital data can help track and monitor policy influence. It will generate new methodologies for mining open social media data, which will help the understanding of the society in social wellbeing inequalities. This can contribute to the strategic roadmap of partner CitizenMe by developing new methodologies for its &quot;next generation human data platform&quot;. The digital footprint measures will complement the traditional economic metrics, which will provide a more accurate indication of policy priorities, hence enhancing the strategic policy making process. It will add inputs to the lo

In [78]:
agnostic_labels_example3 = agnostic_labels[agnostic_labels["project_id"] == validation_example3]

# sort likelihood by "high", "medium", "low" (note needs numeric mapping)
# Create numeric mapping for sorting
likelihood_map = {"high": 3, "medium": 2, "low": 1}

# Sort using the mapping
agnostic_labels_example3 = agnostic_labels_example3.assign(
    likelihood_num=agnostic_labels_example3["likelihood"].map(likelihood_map)
).sort_values("likelihood_num", ascending=False).drop("likelihood_num", axis=1)

display(agnostic_labels_example3)

,project_id,taxonomy_label,likelihood,taxonomy_label_id
83,113BF950-3519-486A-A642-8578B742B755,"Social Sciences > Economics, Econometrics and Finance > Economics and Econometrics > Health Economics and Quality of Life Assessment",high,2d15ef74-a72a-518b-a8aa-776adaafc483
81,113BF950-3519-486A-A642-8578B742B755,Social Sciences > Psychology > Applied Psychology > Digital Mental Health Interventions and Efficacy,medium,c043d5c4-d67e-5ae7-895d-9d6fa720c2e4
86,113BF950-3519-486A-A642-8578B742B755,Social Sciences > Decision Sciences > Management Science and Operations Research > Data Quality Assessment and Improvement,medium,c5b62c38-981b-5e4b-9f98-ea1e6e3d486a
82,113BF950-3519-486A-A642-8578B742B755,Social Sciences > Decision Sciences > Information Systems and Management > Data-Driven Decision Making in Education,low,605a48fb-37b3-5466-b173-7d436e91c8ca
84,113BF950-3519-486A-A642-8578B742B755,Social Sciences > Decision Sciences > Management Science and Operations Research > Methods and Applications of Cluster Analysis in Various Fields,low,36af0d26-13fe-5569-acba-42429ac6a6c2
85,113BF950-3519-486A-A642-8578B742B755,Physical Sciences > Computer Science > Artificial Intelligence > Machine Learning for Internet Traffic Classification,low,bf140b74-f049-56b2-8abd-f6ed1b7c69fb


In [79]:
algorithm_labels_example3 = algorithm_labels[
    algorithm_labels["project_id"] == validation_example3
].sort_values(by="positive", ascending=False)
display(algorithm_labels_example3)

,project_id,taxonomy_label_id,positive,explanation,taxonomy_label
156,113BF950-3519-486A-A642-8578B742B755,05a7d583-6c67-5006-9972-235a5427e8d0,True,"The project focuses on utilizing digital footprints to inform policy-making, directly impacting the economy and society.",Social Sciences > Decision Sciences > Information Systems and Management > Impact of Digital Technologies on Economy and Society
163,113BF950-3519-486A-A642-8578B742B755,684cbb95-a2d3-556c-9f25-2152f610571b,True,The project uses an interdisciplinary approach and involves extensive research collaboration.,Social Sciences > Decision Sciences > Information Systems and Management > Interdisciplinary Research and Collaboration
172,113BF950-3519-486A-A642-8578B742B755,dae44e5f-4c97-587e-80ae-d9b391c5ebe0,True,The project investigates the impact of digitalization on economic systems using digital footprints.,"Social Sciences > Economics, Econometrics and Finance > General Economics, Econometrics and Finance > Impact of Digitalization on Economic Systems"
168,113BF950-3519-486A-A642-8578B742B755,a6da42f4-eb90-5261-b61a-c2156ff221d1,True,The project involves social innovation (digital footprint measures) and interdisciplinary research networks.,Social Sciences > Social Sciences > Sociology and Political Science > Social Innovation and Interdisciplinary Research Networks
167,113BF950-3519-486A-A642-8578B742B755,96be8093-ffff-5652-b906-eccb71534260,True,The project explores the impact of digitalization on economy and society through new measures of life quality from digital footprints.,"Social Sciences > Economics, Econometrics and Finance > General Economics, Econometrics and Finance > Impact of Digitalization on Economy and Society"
166,113BF950-3519-486A-A642-8578B742B755,866a9b98-1bfc-5743-b352-2aa5cdcff42d,True,The project discusses digital data ethics and the regulation of digital footprints.,Social Sciences > Decision Sciences > Information Systems and Management > Regulation and Governance of Emerging Technologies
164,113BF950-3519-486A-A642-8578B742B755,7f841911-b772-5307-8091-3833ac98d10e,True,The project investigates the impact of social media (digital footprints) on societal wellbeing.,Social Sciences > Social Sciences > Sociology and Political Science > Impact of Social Media on Well-being and Behavior
165,113BF950-3519-486A-A642-8578B742B755,80a24828-3e92-5590-a5a5-c61bb99b945e,True,The project involves big data from digital footprints and its impact on society and industry.,Social Sciences > Decision Sciences > Information Systems and Management > Impact of Big Data on Society and Industry
162,113BF950-3519-486A-A642-8578B742B755,4acdc228-c6a5-5614-911e-67369e13fed4,True,The project employs interdisciplinary approaches and spans science and technology.,Social Sciences > Decision Sciences > Management Science and Operations Research > Interdisciplinary Research across Science and Technology
159,113BF950-3519-486A-A642-8578B742B755,244d72c4-bc0c-51c2-9e81-93a6127fc26e,True,The project contributes to strategic policy making by exploring the impact of digital footprints on the economy.,"Social Sciences > Business, Management and Accounting > Strategy and Management > Digital Economy and Sustainable Development"


In [80]:
validation_example_cwts3 = project_scores_cwts[
    project_scores_cwts["project_id"] == validation_example3
]
validation_example_cwts3[
    [
        "taxonomy_label",
        "relevance_score",
        "global_bin",
        "local_bin",
        "confidence_bin",
        "zeroshot_score",
        "zeroshot_bin",
    ]
].sort_values(by="zeroshot_score", ascending=False).head(20)

,taxonomy_label,relevance_score,global_bin,local_bin,confidence_bin,zeroshot_score,zeroshot_bin
99564,Social Sciences > Decision Sciences > Information Systems and Management > Interdisciplinary Research and Collaboration,0.156552,medium,high,medium,0.879716,high
99557,Social Sciences > Decision Sciences > Information Systems and Management > Impact of Digital Technologies on Economy and Society,0.079983,medium,low,low,0.843186,high
99566,Social Sciences > Decision Sciences > Information Systems and Management > Impact of Big Data on Society and Industry,0.090811,high,low,low,0.811751,high
99559,Social Sciences > Decision Sciences > Management Science and Operations Research > Implications of Data Analysis Methods,0.061354,low,low,low,0.811078,high
99560,"Social Sciences > Business, Management and Accounting > Strategy and Management > Digital Economy and Sustainable Development",0.077095,medium,low,low,0.730554,high
99563,Social Sciences > Decision Sciences > Management Science and Operations Research > Interdisciplinary Research across Science and Technology,0.061136,low,low,low,0.696617,medium
99567,Social Sciences > Decision Sciences > Information Systems and Management > Regulation and Governance of Emerging Technologies,0.078042,medium,low,low,0.636099,medium
99561,"Social Sciences > Business, Management and Accounting > Organizational Behavior and Human Resource Management > Human Resource Analytics and Big Data",0.170532,high,high,high,0.438284,low
99575,Social Sciences > Social Sciences > Sociology and Political Science > Social Media Use and Impact on Society,0.172981,high,high,high,0.362461,low
99570,Physical Sciences > Computer Science > Information Systems > Integration of Big Data in Educational Practices,0.085527,high,low,low,0.198219,very low


In [82]:
# Merge the three sets of results
merged_results = (
    # Start with agnostic labels
    agnostic_labels_example3[["taxonomy_label", "likelihood"]]
    # Merge with algorithm labels
    .merge(
        algorithm_labels_example3[["taxonomy_label", "positive"]], 
        on="taxonomy_label",
        how="outer"
    )
    # Merge with zeroshot scores
    .merge(
        validation_example_cwts3[["taxonomy_label", "confidence_bin", "zeroshot_bin"]],
        on="taxonomy_label", 
        how="outer"
    )
)

likelihood_map = {"high": 3, "medium": 2, "low": 1}
merged_results = merged_results.assign(
    likelihood_num=merged_results["likelihood"].map(likelihood_map)
).sort_values(["likelihood_num", "positive"], ascending=[False, False]).drop("likelihood_num", axis=1)

display(merged_results)

,taxonomy_label,likelihood,positive,confidence_bin,zeroshot_bin
14,"Social Sciences > Economics, Econometrics and Finance > Economics and Econometrics > Health Economics and Quality of Life Assessment",high,NaN,NaN,NaN
9,Social Sciences > Decision Sciences > Management Science and Operations Research > Data Quality Assessment and Improvement,medium,NaN,NaN,NaN
17,Social Sciences > Psychology > Applied Psychology > Digital Mental Health Interventions and Efficacy,medium,NaN,NaN,NaN
0,Physical Sciences > Computer Science > Artificial Intelligence > Machine Learning for Internet Traffic Classification,low,NaN,NaN,NaN
4,Social Sciences > Decision Sciences > Information Systems and Management > Data-Driven Decision Making in Education,low,NaN,NaN,NaN
13,Social Sciences > Decision Sciences > Management Science and Operations Research > Methods and Applications of Cluster Analysis in Various Fields,low,NaN,NaN,NaN
3,"Social Sciences > Business, Management and Accounting > Strategy and Management > Digital Economy and Sustainable Development",NaN,True,low,high
5,Social Sciences > Decision Sciences > Information Systems and Management > Impact of Big Data on Society and Industry,NaN,True,low,high
6,Social Sciences > Decision Sciences > Information Systems and Management > Impact of Digital Technologies on Economy and Society,NaN,True,low,high
7,Social Sciences > Decision Sciences > Information Systems and Management > Interdisciplinary Research and Collaboration,NaN,True,medium,high
